1. Pengecekan Integritas & Dimensi Data

      Membaca dataset menggunakan pandas, mengecek jumlah baris/kolom, serta mendeteksi apakah ada nilai yang hilang (missing values / NaN).


In [2]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
print(train.isnull().sum())
print(test.isnull().sum())

(14400, 4) (3600, 3)
id         0
title      0
content    0
label      0
dtype: int64
id         0
title      0
content    0
dtype: int64


2. Analisis Distribusi Target Label

    Menghitung frekuensi kemunculan nilai target (label) pada data latih secara absolut dan persentase.

In [3]:
print(train['label'].value_counts())
print(train['label'].value_counts(normalize=True))

label
1    12960
0     1440
Name: count, dtype: int64
label
1    0.9
0    0.1
Name: proportion, dtype: float64


3. Analisis Pola Frekuensi Teks & Duplikasi (Pola Kunci)

    Mengecek berapa kali suatu judul muncul di data latih dan data uji, lalu melihat kaitannya dengan label.

In [4]:
# Cek berapa kali judul muncul di train
print("Distribusi frekuensi judul di Train:")
print(train['title'].value_counts().value_counts())

# Cek berapa kali judul muncul di Test
print("Distribusi frekuensi judul di Test:")
print(test['title'].value_counts().value_counts())

# Cek label untuk judul yang muncul 1x vs 2x di Train
once_titles = train['title'].value_counts()[train['title'].value_counts() == 1].index
print("Label untuk judul yang muncul 1x:")
print(train[train['title'].isin(once_titles)]['label'].value_counts())

Distribusi frekuensi judul di Train:
count
1    11520
2     1440
Name: count, dtype: int64
Distribusi frekuensi judul di Test:
count
1    2880
2     360
Name: count, dtype: int64
Label untuk judul yang muncul 1x:
label
1    11520
Name: count, dtype: int64


4. Analisis Statistik Panjang Teks

    Menghitung panjang karakter dan jumlah kata pada judul dan isi berita untuk melihat apakah panjang teks bisa menjadi pembeda.

In [5]:
train['title_words'] = train['title'].apply(lambda x: len(x.split()))
train['content_words'] = train['content'].apply(lambda x: len(x.split()))
print(train.groupby('label')[['title_words', 'content_words']].describe().round(1))

      title_words                                       content_words         \
            count mean  std  min  25%   50%   75%   max         count   mean   
label                                                                          
0          1440.0  9.9  2.0  4.0  9.0  10.0  11.0  20.0        1440.0  310.6   
1         12960.0  9.9  2.0  3.0  9.0  10.0  11.0  23.0       12960.0  308.0   

                                                 
         std   min    25%    50%    75%     max  
label                                            
0      172.0  30.0  216.0  271.0  355.0  1694.0  
1      161.0  30.0  218.0  275.0  355.0  4238.0  


Kesimpulan: Panjang kata atau karakter pada pada label 1 dan 0 identik, sehingga panjang teks tidak bisa dijadikan parameter untuk membedakan label. Maka dari itu, yang dapat menjadi pembeda adalah isi dan judul topik itu sendiri atau dengan mengekstrak kata dan menyesuaikan dengan judul.

5. Inspeksi Kualitatif Sampel Data

    Menampilkan contoh nyata dari pasangan judul ganda untuk melihat perbedaan antara pasangan Label 1 vs Label 0.

In [6]:
sample_title = "Ada 70 029 Kasus Suspek Corona Dipantau Per 6 Januari"
print(train[train['title'] == sample_title][['id', 'title', 'content', 'label']])

          id                                              title  \
0          1  Ada 70 029 Kasus Suspek Corona Dipantau Per 6 ...   
11183  11184  Ada 70 029 Kasus Suspek Corona Dipantau Per 6 ...   

                                                 content  label  
0      Terdapat puluhan ribu kasus suspek Corona yang...      1  
11183  Pemerintah melaporkan penambahan kasus baru Co...      0  


6. Feature Engineering

    Mengekstrak fitur kemiripan kata (Word TF-IDF), kemiripan karakter (Char TF-IDF), dan Lexical Overlap:



In [7]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_features(df, tfidf_word=None, tfidf_char=None, is_train=True):
    titles = df['title'].fillna('').astype(str).tolist()
    contents = df['content'].fillna('').astype(str).tolist()
    
    if is_train:
        tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=10000, sublinear_tf=True)
        tfidf_word.fit(titles + contents)
        
        tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=10000, sublinear_tf=True)
        tfidf_char.fit(titles + contents)
    
    t_wvec = tfidf_word.transform(titles)
    c_wvec = tfidf_word.transform(contents)
    cos_word = np.asarray((t_wvec.multiply(c_wvec)).sum(axis=1)).flatten()
    
    t_cvec = tfidf_char.transform(titles)
    c_cvec = tfidf_char.transform(contents)
    cos_char = np.asarray((t_cvec.multiply(c_cvec)).sum(axis=1)).flatten()
    
    feats = []
    for i in range(len(df)):
        t_tokens = titles[i].lower().split()
        c_tokens = contents[i].lower().split()
        t_set, c_set = set(t_tokens), set(c_tokens)
        
        overlap = len(t_set.intersection(c_set))
        t_cov = overlap / (len(t_set) + 1e-9)
        jaccard = overlap / (len(t_set.union(c_set)) + 1e-9)
        
        feats.append([
            cos_word[i],
            cos_char[i],
            t_cov,
            jaccard,
            len(t_tokens),
            len(c_tokens)
        ])
    
    feature_matrix = np.array(feats)
    return feature_matrix, tfidf_word, tfidf_char

print("Mengekstrak fitur...")
X_train, tfidf_word, tfidf_char = extract_features(train, is_train=True)
y_train = train['label'].values
print("Ukuran matriks fitur train:", X_train.shape)

Mengekstrak fitur...
Ukuran matriks fitur train: (14400, 6)


7. Training & Validasi Silang (5-Fold Cross Validation)
    
    Melatih model classifier (Logistic Regression) dan mengevaluasi Macro F1 score

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(train))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    clf = LogisticRegression(C=1.0, max_iter=500)
    clf.fit(X_train[trn_idx], y_train[trn_idx])
    oof_probs[val_idx] = clf.predict_proba(X_train[val_idx])[:, 1]

# Evaluasi dengan Pairwise Ranking
titles_twice = train['title'].value_counts()[train['title'].value_counts() == 2].index
oof_preds_pairwise = np.ones(len(train), dtype=int)

for t in titles_twice:
    rows = train[train['title'] == t]
    idx0, idx1 = rows.index[0], rows.index[1]
    if oof_probs[idx0] >= oof_probs[idx1]:
        oof_preds_pairwise[idx0] = 1
        oof_preds_pairwise[idx1] = 0
    else:
        oof_preds_pairwise[idx0] = 0
        oof_preds_pairwise[idx1] = 1

print("--- HASIL EVALUASI 5-FOLD CV DENGAN PAIRWISE RANKING ---")
print("Macro F1-Score:", f1_score(y_train, oof_preds_pairwise, average='macro'))
print(classification_report(y_train, oof_preds_pairwise, digits=4))

--- HASIL EVALUASI 5-FOLD CV DENGAN PAIRWISE RANKING ---
Macro F1-Score: 0.9938271604938271
              precision    recall  f1-score   support

           0     0.9889    0.9889    0.9889      1440
           1     0.9988    0.9988    0.9988     12960

    accuracy                         0.9978     14400
   macro avg     0.9938    0.9938    0.9938     14400
weighted avg     0.9978    0.9978    0.9978     14400



8. Prediksi Data Uji (test.csv) dan Post-processing

    

In [9]:
# 1. Ekstrak fitur untuk data test
X_test, _, _ = extract_features(test, tfidf_word=tfidf_word, tfidf_char=tfidf_char, is_train=False)
# 2. Latih model final pada seluruh data train
final_model = LogisticRegression(C=1.0, max_iter=500)
final_model.fit(X_train, y_train)
test_probs = final_model.predict_proba(X_test)[:, 1]
# 3. Terapkan aturan Pairwise Ranking
test_preds = np.ones(len(test), dtype=int)
test_titles_twice = test['title'].value_counts()[test['title'].value_counts() == 2].index
for t in test_titles_twice:
    rows = test[test['title'] == t]
    idx0, idx1 = rows.index[0], rows.index[1]
    if test_probs[idx0] >= test_probs[idx1]:
        test_preds[idx0] = 1
        test_preds[idx1] = 0
    else:
        test_preds[idx0] = 0
        test_preds[idx1] = 1
print("Distribusi Prediksi Data Test:")
print(pd.Series(test_preds).value_counts())

Distribusi Prediksi Data Test:
1    3240
0     360
Name: count, dtype: int64


9. Validasi Format

In [10]:
submission = pd.DataFrame({
    'id': test['id'],
    'label': test_preds
})
submission.to_csv('submission_baseline.csv', index=False)
print("Submission berhasil disimpan! Cek 5 baris pertama:")
print(submission.head())
print("\nPengecekan jumlah baris:", submission.shape)

Submission berhasil disimpan! Cek 5 baris pertama:
   id  label
0   1      0
1   2      1
2   3      1
3   4      1
4   5      1

Pengecekan jumlah baris: (3600, 2)
